# NB09 v4 — Adaptive Regulatory Investigation Agent

**Scientific AI Builder | Build04 v4 | Q5 / Q9 / Q10 pilot**

This is an **adaptive agent**, not the QID-specific Lean RAG baseline:

1. LLM **plans** legal research and selects tools and article/annex targets.
2. Python executes local official-snapshot retrieval without another LLM call.
3. A deterministic gate checks **requested evidence availability**, not legal correctness.
4. If important evidence is missing, the LLM **replans once** using observed retrieval failures.
5. LLM synthesizes a concise, cited regulatory assessment.
6. Citation/structure failures may trigger **one final corrective call**.

**Budget:** 2 LLM calls normally, 3 with replanning or repair, at most **4** per question. No live law-checking, no external embedding downloads, and no model-mediated repeated validation.

**Evaluation integrity:** Planner does **not** receive Q1–Q10 gold answers, expected articles, previous model outputs, or historical grades. Q5/Q9/Q10 are development questions already seen by the project team; report score changes as an **in-sample ablation**, not evidence of generalization. Run unseen cases later.

**Important:** This is a frozen five-law research demonstration, **not** a production-ready current-law regulator. Dates in retrieved snapshots do not prove current applicability.

In [5]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

# ---- Configuration: paid calls OFF by default ----
RUN_PAID = False
SELECT_QIDS = ["Q5", "Q9", "Q10"]
MODEL = "gpt-6-luna"          # Change only if API account actually supports this model.
REASONING_EFFORT = "medium"
MAX_LLM_CALLS = 4
MAX_PLAN_TASKS = 6
MAX_RESEARCH_RESULTS_PER_TASK = 2
MAX_TOTAL_CONTEXT_CHARS = 18000
MAX_OUTPUT_TOKENS = 3000
DATA_DIR = __import__("pathlib").Path("data")
OUT_DIR = __import__("pathlib").Path("results/build04/v4_adaptive")
EXPERIMENT_ID = "NB09_build04_adaptive_v4_devpilot"
import os, re, json, hashlib, time, datetime
from pathlib import Path
from getpass import getpass
from collections import Counter
assert DATA_DIR.is_dir(), "Open notebook from Regulatory_agent repository root."
CHUNK_FILE=__import__("pathlib").Path("data/processed/chunks_legal_aware.json")
BENCHMARK_FILE=__import__("pathlib").Path("data/benchmark/benchmark_questions_v1.json")
assert CHUNK_FILE.is_file() and BENCHMARK_FILE.is_file()
chunks=json.loads(CHUNK_FILE.read_text(encoding="utf-8"))
benchmark=json.loads(BENCHMARK_FILE.read_text(encoding="utf-8"))
questions=benchmark["questions"]
assert len(chunks)==1383, f"Unexpected corpus size: {len(chunks)}"
assert set(SELECT_QIDS) <= set(questions)
print("Corpus:",len(chunks),"| Pilot:",SELECT_QIDS,"| Paid:",RUN_PAID)


Corpus: 1383 | Pilot: ['Q5', 'Q9', 'Q10'] | Paid: True


## 1. Frozen legal corpus and tool implementations

The agent can request:

- **exact** lookup for one `Article N`, `Annex [I/II/III/IV]` or numbered row, scoped to a specified regulation.
- **search** for a natural-language question, optionally scoped to a regulation.

The planner's output determines what is retrieved. There is no QID→article lookup table. Strict exact matching falls back to lexical search when no matching article is located. All evidence IDs are derived from the unchanged chunk IDs.

In [6]:
LAW_FILES = {
    "2023/915": "EU_2023_915_consolidated_2026-08-19.html",
    "2019/1793": "EU_2019_1793_consolidated_2026-06-30.html",
    "2017/625": "EU_2017_625_consolidated_2025-01-05.html",
    "178/2002": "EU_2002_178_consolidated_2026-01-01.html",
    "2019/1013": "EU_2019__1013_original.html",
}
ALIASES={"178/2002":"178/2002","2002/178":"178/2002",
         "2023/915":"2023/915","2019/1793":"2019/1793",
         "2017/625":"2017/625","2019/1013":"2019/1013"}
STOPWORDS=set("the and for from into with that this when such where have been under are what which apply about article regulation european union official requirements consignment".split())
def tokens(value):
    return [x for x in re.findall(r"[a-z0-9]+",str(value).lower()) if x not in STOPWORDS and len(x)>1]
def chunk_text(c):
    return " ".join(str(c.get(k) or "") for k in ("regulation_title","article","heading","location","text"))
def e_id(ch):
    return "E"+hashlib.sha256(str(ch.get("chunk_id") or "").encode()).hexdigest()[:10]
def make_evidence(ch, mode):
    law=next((rid for rid,fn in LAW_FILES.items() if ch.get("file")==fn),None)
    assert law is not None, "Unknown frozen document file"
    return {"evidence_id":e_id(ch),"regulation_id":law,"article":ch.get("article"),
            "location":ch.get("location"),"file":ch.get("file"),
            "text":str(ch.get("text") or ""),"retrieved_by":mode,
            "chunk_id":ch.get("chunk_id")}
def parse_law(v):
    if not v:return None
    v=str(v).strip()
    return ALIASES.get(v)
def get_exact(law,provision,limit=2):
    if law not in LAW_FILES:return []
    p=" ".join(str(provision).strip().split())
    file=LAW_FILES[law]
    match_article=re.fullmatch(r"(?:article|art\.)\s*(\d+)(?:\(\d+\))?",p,re.I)
    match_annex=re.fullmatch(r"annex\s+(i|ii|iii|iv|iia)(?:,?\s*section\s*(\d+))?(?:,?\s*row\s*(\d+(?:\.\d+)*))?",p,re.I)
    match_row=re.fullmatch(r"row\s+(\d+(?:\.\d+)*)",p,re.I)
    if not (match_article or match_annex or match_row):return []
    found=[]
    for ch in chunks:
        if ch.get("file")!=file:continue
        loc=str(ch.get("location") or "")
        head=str(ch.get("heading") or "")
        art=str(ch.get("article") or "")
        body=str(ch.get("text") or "")
        if match_article:
            target=match_article.group(1)
            hit=(re.fullmatch(r"Article\s+"+re.escape(target),art,re.I) is not None or
                 re.search(r"(?<![\w])Article\s+"+re.escape(target)+r"(?!\d)",loc,re.I))
        elif match_row:
            row=re.escape(match_row.group(1))
            hit=bool(re.search(r"\bRow:\s*"+row+r"(?![\d.])",body,re.I) or
                     re.search(r"\brow\s+"+row+r"(?![\d.])",loc,re.I))
        else:
            annex=match_annex.group(1).upper()
            row=match_annex.group(3)
            meta=loc+" "+head
            hit=bool(re.search(r"\bAnnex\s+"+re.escape(annex)+r"\b",meta,re.I))
            if row:
                hit=hit and bool(re.search(r"\brow\s+"+re.escape(row)+r"(?![\d.])",meta,re.I) or
                                    re.search(r"\bRow:\s*"+re.escape(row)+r"(?![\d.])",body,re.I))
        if hit:found.append(ch)
    found.sort(key=lambda c:(0 if "logical row" in str(c.get("location")).lower() else 1,-len(str(c.get("text") or ""))))
    return found[:limit]
# Build a lightweight *local* IDF lexical index once (no model download).
DOC_TOKENS=[Counter(tokens(chunk_text(c))) for c in chunks]
DF=Counter(x for d in DOC_TOKENS for x in d)
import math
N=len(chunks)
def keyword_search(query,law=None,limit=2):
    qt=set(tokens(query)); results=[]
    for ch,cnt in zip(chunks,DOC_TOKENS):
        if law and ch.get("file")!=LAW_FILES[law]:continue
        score=sum((1+math.log1p(cnt[t]))*math.log1p((N+1)/(DF[t]+1)) for t in qt if cnt[t])
        if score>0:results.append((score,ch))
    results.sort(key=lambda x:-x[0])
    return [c for _,c in results[:limit]]
def run_tool(task):
    law=parse_law(task.get("regulation_id"))
    mode=task.get("mode","search")
    query=str(task.get("query") or "").strip()
    provision=str(task.get("provision") or "").strip()
    if mode=="exact" and law and provision:
        matches=get_exact(law,provision,MAX_RESEARCH_RESULTS_PER_TASK)
        if matches:return matches,"exact"
        # When exact lookup fails, fall back; record missing exact target.
    q=" ".join(filter(None,[provision,query]))
    return keyword_search(q,law,MAX_RESEARCH_RESULTS_PER_TASK),"lexical"
def collect(tasks,previous=None):
    out={e["evidence_id"]:e for e in (previous or [])}
    events=[];seen=set()
    for t in tasks:
        signature=json.dumps([t.get(k) for k in ("mode","regulation_id","provision","query")],sort_keys=True)
        if signature in seen:continue
        seen.add(signature)
        matching,method=run_tool(t)
        new=0
        for ch in matching:
            e=make_evidence(ch,method)
            if e["evidence_id"] not in out:new+=1
            out.setdefault(e["evidence_id"],e)
        events.append({"task":t,"method":method,"hits":len(matching),"new_evidence":new,
                       "requested_exact_found": bool(method=="exact" and matching)})
    return list(out.values()),events


## 2. Adaptive planning and bounded gap handling

The first LLM call generates a structured tool plan and research objectives, **without providing substantive legal answers**. The deterministic gate identifies empty searches and exact-lookup failures. Replanning is limited to one call, and only triggered by missing critical evidence or unusually thin results; it receives no past model answer or gold reference.

**Limit:** this gate cannot assess whether every legal requirement is covered. Human review remains essential.

In [7]:
PLANNER_PROMPT = '''You are a regulatory investigation planner, not a legal-answer generator.
From the user's EU food-regulation question, choose 2-6 *different* essential legal research tasks.
You have a frozen five-law corpus: 2023/915, 2019/1793, 2017/625, 178/2002, 2019/1013.
For each task choose either:
  mode "exact": one regulation_id and one unambiguous "Article N" or "Annex II" or numbered "row 1.1.3";
  mode "search": natural-language query and optionally a regulation_id.
Use exact whenever a specific source provision can be named with justified confidence.
Do not invent legal rules, results, or cite text you have not seen.
Broad questions require several distinct research domains (contaminants, entry, documentation, responsibility, enforcement).
Return JSON ONLY:
{"tasks":[{"mode":"exact|search","regulation_id":"... or null",
"provision":"... or empty","query":"...","critical":true}],
"research_intentions":["..."]}
Do not return the substantive answer.'''
REPLAN_PROMPT = '''Choose at most TWO new follow-up tool tasks based on missing/failed evidence, not on a hidden answer.
Avoid exact targets already attempted with no matching evidence. Use a different search scope or more specific wording.
Never fabricate a legal obligation. Return JSON only: {"tasks":[...]} with same task schema.'''
ANSWER_PROMPT = '''Produce a concise but operational EU regulatory research answer to the question.
Use ONLY the supplied frozen document excerpts and [E..........] evidence IDs.
Any substantive legal conclusion must cite the specific supporting IDs.
Separate what the frozen texts establish from missing shipment facts and current-law applicability.
A detected contaminant is not necessarily an exceedance. A consolidation date is not verification of present legal effect.
Where evidence for required topics is absent, say exactly what could not be established. Do not fill from memory.
When relevant, map the source provision to the actual compliance obligation, including responsibilities and conditions.
Use exactly these four Markdown headings:
## Short answer
## Evidence-supported requirements
## Facts and law to verify
## Practical next steps
Avoid repetitive disclaimers and unwarranted refusal to state rules evidenced by snapshots.'''
REPAIR_PROMPT = ANSWER_PROMPT + '''
Repair only the enumerated citation/format audit errors. Do not add unsupported claims.'''
def safe_json(s):
    s=s.strip()
    s=re.sub(r"^```(?:json)?\s*|\s*```$","",s,flags=re.I)
    return json.loads(s)
def sanitize_tasks(obj,cap=MAX_PLAN_TASKS):
    clean=[]
    for t in obj.get("tasks",[])[:cap]:
        if not isinstance(t,dict):continue
        mode=t.get("mode","search")
        if mode not in ("exact","search"): mode="search"
        law=parse_law(t.get("regulation_id"))
        provision=str(t.get("provision") or "").strip()[:90]
        query=str(t.get("query") or "").strip()[:250]
        if mode=="exact" and (not law or not provision):mode="search"
        if not query:query=provision or "EU regulatory requirement"
        clean.append({"mode":mode,"regulation_id":law,"provision":provision,
                      "query":query,"critical":bool(t.get("critical",True))})
    return clean
def compact_sources(evidence,max_chars=MAX_TOTAL_CONTEXT_CHARS):
    out=[];used=0
    for e in evidence:
        room=min(1600,max_chars-used)
        if room<200:break
        t=e["text"]
        snippet=t if len(t)<=room else t[:int(room*.75)]+" [...] "+t[-int(room*.25):]
        out.append({"id":e["evidence_id"],"regulation":e["regulation_id"],
                    "article":e.get("article"),"location":e.get("location"),"text":snippet})
        used+=len(snippet)
    return out
def audit_answer(answer,evidence):
    allowed={e["evidence_id"] for e in evidence}
    cited=set(re.findall(r"\[(E[0-9a-f]{10})\]",answer))
    incorrect=sorted(cited-allowed)
    headings=["## Short answer","## Evidence-supported requirements",
              "## Facts and law to verify","## Practical next steps"]
    absent=[h for h in headings if h not in answer]
    return {"passed":bool(cited) and not incorrect and not absent,
            "unknown_evidence_ids":incorrect,"missing_headings":absent,
            "cited_evidence_count":len(cited),
            "limitation":"Syntactic citation audit; not independent legal entailment validation."}


## 3. API call budget and execution

Set `RUN_PAID=True` *only* after inspecting dry-run corpus and configuration. The notebook writes a new immutable per-QID file under `data/build04_nb09_v4_adaptive_runs/`.

An API key is requested securely at runtime; no API calls run in dry-run mode. The output includes raw plan, retrieval and gaps, model usage, timing and answer for later blinded human scoring.

In [8]:
# Paid calls happen only in this cell when enabled.
def call_model(client,instructions,payload,log, max_tokens=MAX_OUTPUT_TOKENS):
    assert len(log)<MAX_LLM_CALLS,"Per-question LLM budget exceeded"
    resp=client.responses.create(
        model=MODEL,
        reasoning={"effort":REASONING_EFFORT},
        instructions=instructions,
        input=json.dumps(payload,ensure_ascii=False),
        max_output_tokens=max_tokens,store=False,
    )
    usage=getattr(resp,"usage",None)
    log.append({"input_tokens":int(getattr(usage,"input_tokens",0) or 0),
                "output_tokens":int(getattr(usage,"output_tokens",0) or 0)})
    value=getattr(resp,"output_text","") or ""
    if not value.strip():
        value="\n".join(str(getattr(c,"text","")) for block in getattr(resp,"output",[])
                        for c in getattr(block,"content",[]))
    if not value.strip():raise RuntimeError("Empty model response")
    return value.strip()
def run_adaptive(qid,client):
    log=[];start=time.perf_counter()
    question=questions[qid]
    case_context="Türkiye-origin dried figs with aflatoxin B1 detected; concentration, product details and import/market stage unknown."
    raw_plan=call_model(client,PLANNER_PROMPT,{"question":question,"case_context":case_context},log,1800)
    plan=sanitize_tasks(safe_json(raw_plan))
    if not plan:raise RuntimeError("No valid tool plan")
    evidence,events=collect(plan)
    replanned=False; extra_plan=[]
    # Only a bounded retry when critical requested evidence is missing or overall evidence is thin.
    gaps=[e for e in events if e["task"].get("critical") and
          (not e["hits"] or (e["task"]["mode"]=="exact" and not e["requested_exact_found"]))]
    if (gaps or len(evidence)<3) and len(log)<MAX_LLM_CALLS-1:
        retry_raw=call_model(client,REPLAN_PROMPT,
            {"question":question,"initial_tasks":plan,
             "failed_or_fallback_retrieval":gaps,"source_locations":[
                 {"regulation":e["regulation_id"],"article":e["article"],"location":e["location"]}
                 for e in evidence[:12]]},log,1100)
        extra_plan=sanitize_tasks(safe_json(retry_raw),cap=2)
        if extra_plan:
            evidence,extra_events=collect(extra_plan,evidence)
            events+=extra_events
            replanned=True
    if not evidence:raise RuntimeError("No legal evidence available: abort, do not fabricate")
    answer=call_model(client,ANSWER_PROMPT,{"question":question,"case_context":case_context,
          "sources":compact_sources(evidence),"retrieval_gaps":gaps},log,MAX_OUTPUT_TOKENS)
    audit=audit_answer(answer,evidence);repaired=False
    if not audit["passed"] and len(log)<MAX_LLM_CALLS:
        answer=call_model(client,REPAIR_PROMPT,{"question":question,"case_context":case_context,
            "sources":compact_sources(evidence),"draft":answer,"audit":audit},log,MAX_OUTPUT_TOKENS)
        audit=audit_answer(answer,evidence);repaired=True
    return {"qid":qid,"question":question,"experiment_id":EXPERIMENT_ID,
        "plan":plan,"replan":extra_plan,"replanned":replanned,"retrieval_events":events,
        "evidence":evidence,"final_output":answer,"audit":audit,
        "repair_used":repaired,"llm_calls":len(log),
        "usage":{"input_tokens":sum(x["input_tokens"] for x in log),
                 "output_tokens":sum(x["output_tokens"] for x in log)},
        "latency_seconds":round(time.perf_counter()-start,2),
        "search_calls":len(events),"status":"pending_independent_human_review",
        "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat()}
SETTINGS={"experiment_id":EXPERIMENT_ID,"model":MODEL,"reasoning_effort":REASONING_EFFORT,
          "max_llm_calls":MAX_LLM_CALLS,"benchmark_sha256":hashlib.sha256(BENCHMARK_FILE.read_bytes()).hexdigest(),
          "chunks_sha256":hashlib.sha256(CHUNK_FILE.read_bytes()).hexdigest(),
          "adaptive_research":True,"case_study_exposure":"in_sample_dev",
          "official_live_law_check":False}
OUT_DIR.mkdir(parents=True,exist_ok=True)
manifest=OUT_DIR/"manifest.json"
if RUN_PAID:
    if manifest.exists():
        assert json.loads(manifest.read_text(encoding="utf-8"))==SETTINGS, "Manifest mismatch: use new experiment directory."
    else:
        with manifest.open("x",encoding="utf-8") as f:json.dump(SETTINGS,f,ensure_ascii=False,indent=2)
    from openai import OpenAI
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"]=getpass("OPENAI_API_KEY (hidden): ").strip()
    client=OpenAI()
    for qid in SELECT_QIDS:
        out=OUT_DIR/f"{qid}.json"
        if out.exists():
            print("SKIP existing result",qid);continue
        result=run_adaptive(qid,client)
        result["settings"]=SETTINGS
        with out.open("x",encoding="utf-8") as f:json.dump(result,f,ensure_ascii=False,indent=2)
        print(qid,"saved | LLM",result["llm_calls"],"| tokens",result["usage"],
              "| seconds",result["latency_seconds"],"| audit",result["audit"]["passed"])
else:
    print("DRY RUN — no LLM calls. Ready for",SELECT_QIDS)


RuntimeError: Empty model response

## 4. Compare Lean RAG vs Adaptive v4 vs Full Investigation v3

Requires independently generated outputs in their respective folders. **Correct / Grounded / Complete are not inferred automatically** from citation audits, search counts or token usage. Grade against the unchanged frozen Verified Reference, with sources and model names hidden if possible.

Record improvements on Q5/Q9/Q10 as *development-set* results only. Add new unseen cases before making generalization claims.

In [ ]:
rows=[]
for qid in SELECT_QIDS:
    for label,folder in [
        ("Full Investigation v3","build04_nb09_v3_runs"),
        ("Lean RAG v4","build04_nb09_v4_lean_runs"),
        ("Adaptive Agent v4","build04_nb09_v4_adaptive_runs")]:
        fp=__import__("pathlib").Path({"build04_nb09_v3_runs":"results/build04","build04_nb09_v4_lean_runs":"results/build04/v4_lean","build04_nb09_v4_adaptive_runs":"results/build04/v4_adaptive"}[folder])/f"{qid}.json"
        r=json.loads(fp.read_text(encoding="utf-8")) if fp.exists() else {}
        u=r.get("usage") or {}
        rows.append({"qid":qid,"variant":label,"available":bool(r),
                     "llm_calls":r.get("llm_calls"),
                     "input_tokens":u.get("input_tokens"),
                     "output_tokens":u.get("output_tokens"),
                     "wall_seconds":r.get("latency_seconds"),
                     "search_or_lookup_count":r.get("search_calls",r.get("local_provision_lookups")),
                     "citation_audit_pass":(r.get("audit") or {}).get("passed"),
                     "Correct":"", "Grounded":"","Complete":""})
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for row in rows:print(row)
